In [ ]:
# Cell 0 — Install dependencies + clone feat/corpo branch (private repo)
# Pins:
#   trl>=0.21 — earlier versions import GuidedDecodingParams which vllm 0.21+ removed
#   vllm>=0.8 — TRL 0.21+ assumes the newer vllm API surface
!pip install -q "torchao>=0.16.0" "trl>=0.21.0" "peft>=0.13.0" "vllm>=0.8.0" "openai>=1.0.0" unsloth liger-kernel datasets transformers accelerate numpy

from google.colab import userdata
GITHUB_PAT = userdata.get('GITHUB_PAT')

# Clean any prior failed clone, then clone the feat/corpo branch
!rm -rf /content/sft
!git clone --branch feat/corpo --depth 1 \
    https://{GITHUB_PAT}@github.com/deepanathanrajendiran-hub/sft-code-review.git \
    /content/sft

# Copy Python files + tests to /content/
!cp /content/sft/*.py /content/sft/pyproject.toml /content/
!cp -r /content/sft/tests /content/

# Verify
import os; os.chdir("/content")
!ls /content/corpo_*.py /content/swecare_*.py /content/ood_metrics.py
print("\nVersion check:")
import trl, vllm
print(f"  trl   : {trl.__version__}")
print(f"  vllm  : {vllm.__version__}")

In [ ]:
# Cell 1 — Mount Drive, load secrets, verify v4 backup
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'

# v4 adapter paths — USER MUST ensure backup exists before this cell runs
V4_ADAPTER = '/content/drive/MyDrive/sft/code-reviewer-lora-v4-traces'
V4_BACKUP  = '/content/drive/MyDrive/sft/code-reviewer-lora-v4-traces-backup'
assert os.path.exists(V4_BACKUP + '/adapter_config.json'), \
    f"v4 backup missing! Create one BEFORE running: cp -r {V4_ADAPTER} {V4_BACKUP}"
print(f"v4 adapter:  {V4_ADAPTER}")
print(f"v4 backup:   {V4_BACKUP}")

In [ ]:
# Cell 2 — Load SWE-CARE dev (train) and test (eval), build base-sample cache (~5-10 min, $0.05)
# dev=7086 rows for CoRPO training; test=632 (after repo filter) reserved for OOD eval
import json, os, random

# Generate training prompts from dev split
!python /content/swecare_loader.py \
    --split dev \
    --output /content/ood_dev_prompts_raw.jsonl \
    --train-jsonl /content/drive/MyDrive/sft/train_dataset_clean.jsonl

# Sample 1500 dev rows (seeded) for actual training
with open('/content/ood_dev_prompts_raw.jsonl') as f:
    dev_rows = [json.loads(l) for l in f if l.strip()]
sample = random.Random(42).sample(dev_rows, min(1500, len(dev_rows)))
with open('/content/ood_train_prompts.jsonl', 'w') as f:
    for r in sample: f.write(json.dumps(r) + '\n')
print(f"dev pool: {len(dev_rows)}   training sample: {len(sample)}")

# Generate eval input from test split (separate, no overlap)
!python /content/swecare_loader.py \
    --split test \
    --output /content/ood_input.jsonl \
    --train-jsonl /content/drive/MyDrive/sft/train_dataset_clean.jsonl

# Diagnostic: how much repo-level overlap between training (dev) and eval (test)?
import json
def _read_repos(path):
    with open(path) as f:
        return {json.loads(l)['repo'] for l in f if l.strip()}

train_repos = _read_repos('/content/ood_train_prompts.jsonl')
eval_repos = _read_repos('/content/ood_input.jsonl')
overlap = train_repos & eval_repos
unique_to_eval = eval_repos - train_repos
print(f"[cell2] train repos:    {len(train_repos)}")
print(f"[cell2] eval repos:     {len(eval_repos)}")
print(f"[cell2] overlap:        {len(overlap)}/{len(eval_repos)} eval repos also in train ({100*len(overlap)/max(len(eval_repos),1):.0f}%)")
print(f"[cell2] eval-only:      {len(unique_to_eval)}")
print(f"[cell2] interpretation: {'high overlap — `novel PR in mostly-shared repos` measurement' if len(overlap)/max(len(eval_repos),1) > 0.5 else 'mostly-disjoint repos — closer to true OOD'}")

# Skip base cache build if cache exists AND covers every prompt in the current sample
import os, json
cache_path = '/content/cache/base_samples.jsonl'
build_cache = True
if os.path.exists(cache_path):
    with open(cache_path) as f:
        cached_ids = {json.loads(l)['instance_id'] for l in f if l.strip()}
    sample_ids = {r['instance_id'] for r in sample}
    missing = sample_ids - cached_ids
    if not missing:
        print(f"[cell2] base cache covers all {len(sample_ids)} sample prompts — skipping")
        build_cache = False
    else:
        print(f"[cell2] base cache missing {len(missing)} prompts (e.g. {next(iter(missing))!r}) — rebuilding")

if build_cache:
    os.system(
        'python /content/corpo_reward.py --build-base-cache '
        f'--input /content/ood_train_prompts.jsonl '
        f'--output {cache_path} '
        '--base-model unsloth/Qwen2.5-Coder-7B-Instruct'
    )

In [ ]:
# Cell 3 — Pre-training variance gate (~10 min, $0.40)
# This Python cell raises CalledProcessError if the gate fails, blocking Cell 4 from running.
import subprocess
subprocess.check_call([
    "python", "/content/corpo_train.py", "--variance-gate-only",
    "--v4-adapter", V4_ADAPTER,
    "--v4-backup", V4_BACKUP,
    "--train-prompts", "/content/ood_train_prompts.jsonl",
    "--base-cache", "/content/cache/base_samples.jsonl",
    "--output-dir", "/content/corpo-out",
])
print("[variance-gate] PASSED — safe to proceed to Cell 4")

In [ ]:
# Cell 4 — Train CoRPO (~4-6 hours)
# If Colab disconnects mid-training, RESUME by adding `--resume /content/corpo-out/checkpoint-NNN` 
# (find the latest checkpoint with: !ls -t /content/corpo-out/checkpoint-* | head -1)
!python /content/corpo_train.py \
    --v4-adapter {V4_ADAPTER} \
    --v4-backup {V4_BACKUP} \
    --train-prompts /content/ood_train_prompts.jsonl \
    --base-cache /content/cache/base_samples.jsonl \
    --output-dir /content/corpo-out \
    --r-min-correct 0.5 \
    --kl-beta 0.01 \
    --learning-rate 5e-6 \
    --num-generations 8 \
    --prompts-per-step 4 \
    --max-new-tokens 2048 \
    --epochs 1 \
    --checkpoint-every 50 \
    --copy-to /content/drive/MyDrive/sft/code-reviewer-lora-v4-corpo

In [ ]:
# Cell 5 — Verify chat_template parity, merge corpo adapter, generate predictions on 632 OOD set

# 5a. Verify v4 chat_template matches base — if not, run_ood_eval.py's assert will fire mid-run
from transformers import AutoTokenizer
v4_tok = AutoTokenizer.from_pretrained(V4_ADAPTER)
base_tok = AutoTokenizer.from_pretrained('unsloth/Qwen2.5-Coder-7B-Instruct')
assert v4_tok.chat_template == base_tok.chat_template, \
    "v4 chat_template differs from base — patch run_ood_eval.py:135 to load tokenizer per-model"
del v4_tok, base_tok
print("[cell5] chat_template parity: OK")

# 5b. Merge corpo adapter (for vLLM eval — vLLM expects a full model)
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    'unsloth/Qwen2.5-Coder-7B-Instruct', dtype=torch.bfloat16
)
peft_model = PeftModel.from_pretrained(base, '/content/corpo-out/final')
merged = peft_model.merge_and_unload()
merged.save_pretrained('/content/sft-v4corpo-merged-for-eval', safe_serialization=True)
AutoTokenizer.from_pretrained(V4_ADAPTER).save_pretrained('/content/sft-v4corpo-merged-for-eval')

del merged, peft_model, base
gc.collect(); torch.cuda.empty_cache()

# 5c. Generate v4-corpo predictions ONLY on the 632 OOD set (--skip-base saves ~30 min)
!python /content/run_ood_eval.py \
    --input /content/ood_input.jsonl \
    --output /content/ood_preds_v4corpo.jsonl \
    --v4-model /content/sft-v4corpo-merged-for-eval \
    --skip-base

In [ ]:
# Cell 6 — V4-Pro 3-vote re-baseline + post-corpo eval + Haiku cross-check + decision gate
import json

# 1. Recompute halluc_v4 from baseline preds under V4-Pro (skip pairwise — gate only needs halluc + per-domain)
!python /content/ood_metrics.py \
    --preds /content/drive/MyDrive/sft/ood_preds_v4.jsonl \
    --labels /content/ood_input.jsonl \
    --judge v4pro3vote \
    --skip-pairwise \
    --output /content/v4_baseline_v4pro.json

# 2. Merge v4_pred (from baseline preds) into v4corpo preds for the corpo-vs-v4 comparison
def _merge_preds(v4_path, corpo_path, out_path):
    v4_lookup = {}
    with open(v4_path) as f:
        for line in f:
            r = json.loads(line)
            v4_lookup[r['instance_id']] = r['v4_pred']
    with open(corpo_path) as f, open(out_path, 'w') as out:
        for line in f:
            c = json.loads(line)
            iid = c['instance_id']
            c['v4corpo_pred'] = c.pop('v4_pred', '')  # corpo run wrote it as v4_pred
            c['v4_pred'] = v4_lookup.get(iid, '')
            out.write(json.dumps(c) + '\n')

_merge_preds('/content/drive/MyDrive/sft/ood_preds_v4.jsonl',
             '/content/ood_preds_v4corpo.jsonl',
             '/content/ood_preds_merged.jsonl')

# 3. v4-corpo vs v4 under V4-Pro 3-vote (THE measurement)
!python /content/ood_metrics.py \
    --preds /content/ood_preds_merged.jsonl \
    --labels /content/ood_input.jsonl \
    --judge v4pro3vote \
    --pred-a-field v4corpo_pred \
    --pred-b-field v4_pred \
    --output /content/corpo_vs_v4_v4pro.json

# 4. Haiku cross-check on 100-prompt subset (Goodhart guard)
# Sample 100 random rows (seeded) for Haiku cross-check (NOT first 100 — that's grouped by dataset order)
import random
with open('/content/ood_preds_merged.jsonl') as f:
    all_rows = [l.strip() for l in f if l.strip()]
subset = random.Random(42).sample(all_rows, min(100, len(all_rows)))
with open('/content/ood_preds_merged_100.jsonl', 'w') as f:
    for r in subset: f.write(r + '\n')
!python /content/ood_metrics.py \
    --preds /content/ood_preds_merged_100.jsonl \
    --labels /content/ood_input.jsonl \
    --judge haiku \
    --pred-a-field v4corpo_pred \
    --pred-b-field v4_pred \
    --output /content/corpo_vs_v4_haiku.json

# 5. Decision gate verdict
!python /content/corpo_decision_gate.py \
    --v4-baseline-json /content/v4_baseline_v4pro.json \
    --corpo-eval-json  /content/corpo_vs_v4_v4pro.json \
    --haiku-cross-check-json /content/corpo_vs_v4_haiku.json \
    --variance-gate-passed